# Main Result: PID Decomposition

### Matched 12-analyte panel: the cross-database comparison without the confound

Protocol: fixed
pre-collapse grid, k=13, 5 k-means seeds, BROJA primary with CCS sensitivity,
permutation nulls at seed 0. Section 1 re-checks the convergence plateau on the
new 24-d channels rather than assuming k=13 still sits inside it.

The panels are matched across two datasets (MIMIC-IV and eICU), 12-analyte panel. With the same analytes, we control the amount of information between input and outcomes.
Therefore, we can compare the information decomposition results across the two datasets without the confound of different analytes.

In [1]:
import sys
sys.path.insert(0, "../src")

import json, time
import numpy as np
import pandas as pd

import mimic_channels as mimic
import eicu_channels as eicu
import quantize as qz
import pid_broja as pb

pd.set_option("display.width", 220)

K, SEEDS = 13, [0, 1, 2, 3, 4]
ATOMS = ("red", "u_val", "u_str", "syn")
N_PERM, N_JOBS, BUDGET_S = 1000, 16, 90.0
TARGETS = ["mortality", "gender_female", "age_ge65", "race_binary"]

MG_MIMIC = 50960
PANEL12_EICU = {"bicarbonate", "chloride", "creatinine", "glucose", "Hct", "Hgb",
                "platelets x 1000", "potassium", "RBC", "sodium", "BUN", "WBC x 1000"}

cc_m = mimic.complete_case(itemid_scope=set(mimic.CORE) - {MG_MIMIC})
cc_e = eicu.complete_case(labname_scope=PANEL12_EICU)
DBS = [("MIMIC-IV", mimic, cc_m), ("eICU-CRD", eicu, cc_e)]
for db, _, cc in DBS:
    print(f"{db:9s} n={cc.meta['n']:7,d}  V={cc.Xv.shape[1]}d S={cc.Xs.shape[1]}d  "
          f"prev={cc.meta['prevalence']:.4f}  dropped={cc.meta['drop_rate']:.1%}")

def H(y):
    p = float(np.mean(y)); return -(p*np.log(p) + (1-p)*np.log(1-p))

def grid(ch, mod, k, seed):
    v, _ = qz.kmeans_levels(qz.standardize_for_kmeans(ch.Xv), k, seed=seed)
    s, _ = qz.kmeans_levels(qz.standardize_for_kmeans(ch.Xs, sentinel=mod.TIME_SENTINEL), k, seed=seed)
    return v, s

MIMIC-IV  n= 64,623  V=24d S=24d  prev=0.1069  dropped=1.1%
eICU-CRD  n=120,266  V=24d S=24d  prev=0.0947  dropped=12.7%


## 1. How many K in discretization step is appropriate?

In [2]:
# Full k sweep, both arms. `fixed` is the pre-collapse k x k grid; `collapse` is
# the same grid with collapse_for_occupancy applied, so the only difference
# between the arms is the occupancy gate. Mortality, full cohort, both databases.
#
# k is chosen on where the atoms converge and the seed spread stops narrowing --
# never on the reported statistic. Picking k by whichever value maximises syn%
# would be selection bias, since plug-in MI rises monotonically with resolution.
K_GRID = [4, 5, 6, 7, 8, 10, 13, 16, 20]

rows = []
for db, mod, cc in DBS:
    for k in K_GRID:
        acc = {"fixed": {a: [] for a in ATOMS}, "collapse": {a: [] for a in ATOMS}}
        empt, fails = [], {"fixed": 0, "collapse": 0}
        for sd in SEEDS:
            v, s = grid(cc, mod, k, sd)
            d = pb.decompose(v, s, cc.y, measure="broja")
            empt.append(d["n_empty_cells"])
            if d["converged"]:
                for a in ATOMS: acc["fixed"][a].append(d["atoms_pct_of_joint"][a])
            else:
                fails["fixed"] += 1
            v2, s2, _ = qz.collapse_for_occupancy(v, s, cc.y)
            dc = pb.decompose(v2, s2, cc.y, measure="broja")
            if dc["converged"]:
                for a in ATOMS: acc["collapse"][a].append(dc["atoms_pct_of_joint"][a])
            else:
                fails["collapse"] += 1
        for arm in ("fixed", "collapse"):
            got = {a: np.array(acc[arm][a], float) for a in ATOMS}
            n_ok = len(got["syn"])
            rec = {"db": db, "k": k, "arm": arm, "cells": k*k*2, "n_seeds_ok": n_ok,
                   "n_seeds_failed": fails[arm], "empty": float(np.mean(empt))}
            for a in ATOMS:
                rec[f"{a}_mean"] = float(got[a].mean()) if n_ok else float("nan")
                rec[f"{a}_sd"] = float(got[a].std()) if n_ok > 1 else float("nan")
            rec["worst_sd"] = (float(np.nanmax([rec[f"{a}_sd"] for a in ATOMS]))
                               if n_ok > 1 else float("nan"))
            rows.append(rec)

sweep = pd.DataFrame(rows)
plateau = sweep[sweep.arm == "fixed"].drop(columns=["arm"]).reset_index(drop=True)
print("mortality, BROJA, fixed grid, 5 seeds, 12-analyte panel\n")
print(plateau[["db","k","cells","empty","n_seeds_ok","red_mean","red_sd",
               "u_val_mean","u_val_sd","syn_mean","syn_sd","worst_sd"]]
      .to_string(index=False, float_format=lambda x: f"{x:.2f}"))

mortality, BROJA, fixed grid, 5 seeds, 12-analyte panel

      db  k  cells  empty  n_seeds_ok  red_mean  red_sd  u_val_mean  u_val_sd  syn_mean  syn_sd  worst_sd
MIMIC-IV  4     32   0.00           5      9.55   10.73       70.99     24.18     19.00   13.35     24.18
MIMIC-IV  5     50   0.00           5     18.84    7.86       49.26     18.21     25.66    6.43     18.21
MIMIC-IV  6     72   0.00           5     23.21    1.72       39.81      7.56     31.82    4.36      7.56
MIMIC-IV  7     98   0.00           5     23.97    1.51       42.24      4.53     30.23    1.77      4.53
MIMIC-IV  8    128   0.00           5     22.89    3.03       39.34      8.20     31.63    2.54      8.20
MIMIC-IV 10    200   2.00           5     24.73    1.36       35.70      6.70     32.50    4.20      6.70
MIMIC-IV 13    338   7.40           5     24.54    2.12       39.78      2.65     31.51    2.71      2.71
MIMIC-IV 16    512  10.20           4     24.09    1.12       42.40      3.57     30.25    3.35

### Where the atoms converge

`worst_sd` is the largest per-atom standard deviation across the five k-means
seeds. It is the quantity k is chosen on.

In [3]:
fx = sweep[sweep.arm == "fixed"]
piv = fx.pivot(index="k", columns="db", values="worst_sd")
print("seed spread (worst_sd) against k\n")
print(piv.to_string(float_format=lambda x: f"{x:.2f}"))
print(f"\n  minimum at k={int(piv.idxmin().mode().iloc[0])} for "
      f"{(piv.idxmin() == piv.idxmin().mode().iloc[0]).sum()}/{piv.shape[1]} databases")
deg = fx[fx.n_seeds_failed > 0]
if len(deg):
    print("\n  solver degeneracy (a reason not to reach past k=13):")
    for _, r in deg.iterrows():
        print(f"    {r.db:9s} k={int(r.k):2d}: {int(r.n_seeds_failed)}/{len(SEEDS)} seeds fail, "
              f"{r.empty:.1f} empty cells on average")
print(f"\n  atoms at k={K}:")
for _, r in fx[fx.k == K].iterrows():
    print(f"    {r.db:9s} " + "  ".join(f"{a}={r[a+'_mean']:5.2f}" for a in ATOMS))

seed spread (worst_sd) against k

db  MIMIC-IV  eICU-CRD
k                     
4      24.18      6.92
5      18.21      7.25
6       7.56      3.93
7       4.53      4.99
8       8.20      6.65
10      6.70      4.62
13      2.71      4.52
16      3.57      6.03
20      4.04      2.60

  minimum at k=13 for 1/2 databases

  solver degeneracy (a reason not to reach past k=13):
    MIMIC-IV  k=16: 1/5 seeds fail, 0.0 empty cells on average

  atoms at k=13:
    MIMIC-IV  red=24.54  u_val=39.78  u_str= 4.16  syn=31.51
    eICU-CRD  red=19.97  u_val=49.10  u_str= 0.14  syn=30.79


### What `collapse_for_occupancy` does

Both arms share the same k-means grid at each (k, seed), so any difference is
attributable to the occupancy gate and nothing else.

In [4]:
cmp = (sweep.pivot_table(index=["db","k"], columns="arm", values=["u_val_mean","syn_mean","worst_sd"])
       .reset_index())
cmp.columns = ["db","k","syn_collapse","syn_fixed","u_val_collapse","u_val_fixed",
               "sd_collapse","sd_fixed"]
cmp = cmp[["db","k","u_val_fixed","u_val_collapse","syn_fixed","syn_collapse","sd_fixed","sd_collapse"]]
print(cmp.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

diverge = cmp[(cmp.syn_fixed - cmp.syn_collapse).abs() > 1.0]
k_first = int(diverge.k.min()) if len(diverge) else None
worse = cmp[cmp.sd_collapse > cmp.sd_fixed]
hi = cmp[cmp.k >= K]
print(f"\n  arms first diverge by >1 synergy point at k={k_first}")
print(f"  collapse gives a LARGER seed spread in {len(worse)}/{len(cmp)} (db,k) combinations")
print(f"  at k>={K} it shifts u_val by {(hi.u_val_collapse - hi.u_val_fixed).mean():+.1f} "
      f"and synergy by {(hi.syn_collapse - hi.syn_fixed).mean():+.1f} points")

      db  k  u_val_fixed  u_val_collapse  syn_fixed  syn_collapse  sd_fixed  sd_collapse
MIMIC-IV  4        70.99           70.99      19.00         19.00     24.18        24.18
MIMIC-IV  5        49.26           49.26      25.66         25.66     18.21        18.21
MIMIC-IV  6        39.81           39.81      31.82         31.82      7.56         7.56
MIMIC-IV  7        42.24           44.76      30.23         30.13      4.53         2.85
MIMIC-IV  8        39.34           37.08      31.63         31.63      8.20        20.92
MIMIC-IV 10        35.70           48.29      32.50         19.90      6.70        28.33
MIMIC-IV 13        39.78           62.65      31.51         17.86      2.71        18.24
MIMIC-IV 16        42.40           97.82      30.25          1.29      3.57         1.66
MIMIC-IV 20        36.83           86.92      33.89          4.49      4.04        16.28
eICU-CRD  4        57.16           57.16      23.53         23.53      6.92         6.92
eICU-CRD  5        49

### Save the k-resolution evidence

In [5]:
k_payload = {
    "design": (f"BROJA, mortality, % of I_joint, mean +- sd over {len(SEEDS)} k-means seeds. "
               f"'fixed' = pre-collapse k x k grid; 'collapse' = the same grid with "
               f"collapse_for_occupancy applied. Seeds whose solver cascade fails are excluded "
               f"from the mean and counted in n_seeds_failed. Matched 12-analyte panel, both "
               f"databases."),
    "provenance": "notebooks/02_main_result.ipynb, section 1 (merged from the former 06_k_resolution).",
    "k_grid": K_GRID,
    "seeds": SEEDS,
    "k_chosen": K,
    "sweep": rows,
    "worst_sd_by_k": {db: {int(r.k): (None if np.isnan(r.worst_sd) else round(r.worst_sd, 3))
                           for _, r in fx[fx.db == db].iterrows()} for db in fx.db.unique()},
    "degeneracy": [{"db": r.db, "k": int(r.k), "n_seeds_failed": int(r.n_seeds_failed),
                    "empty_cells": round(r.empty, 1)} for _, r in fx[fx.n_seeds_failed > 0].iterrows()],
    "collapse_verdict": {
        "first_k_diverging_by_1pt_synergy": k_first,
        "n_combinations_where_collapse_has_larger_seed_spread": int(len(worse)),
        "n_combinations_total": int(len(cmp)),
        "mean_u_val_shift_at_k_ge_chosen": round(float((hi.u_val_collapse - hi.u_val_fixed).mean()), 2),
        "mean_syn_shift_at_k_ge_chosen": round(float((hi.syn_collapse - hi.syn_fixed).mean()), 2),
    },
}
with open("../results/k_resolution_headline_impact.json", "w") as f:
    json.dump(k_payload, f, indent=2, default=float)
print("wrote results/k_resolution_headline_impact.json")

wrote results/k_resolution_headline_impact.json


## 2. The matched-panel decomposition

In [6]:
demo_cache = {db: mod.demographic_targets()[0] for db, mod, _ in DBS}

# Perform one-time grid fitting for each database and seed, and cache the results in CODEBOOK.

CODEBOOK = {}
for _db, _mod, _cc in DBS:
    for _sd in SEEDS:
        CODEBOOK[(_db, _sd)] = grid(_cc, _mod, K, _sd)
print(f"codebooks: {len(CODEBOOK)} = {len(DBS)} databases x {len(SEEDS)} seeds, "
      f"each fit on the full cohort")


def target_mask(db, cc, target):
    """Row mask into the full cohort for a target, or None for mortality."""
    if target == "mortality":
        return None
    return demo_cache[db][target].reindex(pd.Index(cc.groups)).notna().to_numpy()


def run(db, mod, cc, target, measure):
    ch = cc if target == "mortality" else mod.retarget(cc, demo_cache[db][target], target)
    keep = target_mask(db, cc, target)
    acc = {a: [] for a in ("red", "u_val", "u_str", "syn")}
    ijs, conv = [], []
    for sd in SEEDS:
        v, s = CODEBOOK[(db, sd)]
        if keep is not None: v, s = v[keep], s[keep]
        d = pb.decompose(v, s, ch.y, measure=measure)
        for a in acc: acc[a].append(d["atoms_pct_of_joint"][a])
        ijs.append(d["mi_plugin_nats"]["I_joint"]); conv.append(d["converged"])
    v, s = CODEBOOK[(db, SEEDS[0])]
    if keep is not None: v, s = v[keep], s[keep]
    d0 = pb.decompose(v, s, ch.y, measure=measure)
    null, n_used = pb.permutation_null_budgeted(v, s, ch.y, n_perm=N_PERM, n_jobs=N_JOBS,
                                                budget_s=BUDGET_S, seed=0, measure=measure)
    syn0, ij0 = d0["atoms_nats"]["syn"], d0["mi_plugin_nats"]["I_joint"]
    ij_null = sum(null[a] for a in ("red", "u_val", "u_str", "syn"))

    keep = np.isfinite(ij_null) & np.isfinite(null["syn"])
    n_eff = int(keep.sum())
    n_drop = int(n_used - n_eff)
    syn_null_f, ij_null_f = null["syn"][keep], ij_null[keep]
    hy = H(ch.y)
    return {"db": db, "target": target, "measure": measure, "n": int(len(ch.y)),
            "prevalence": float(ch.y.mean()), "H_Y": hy,
            "I_joint_mean": float(np.mean(ijs)), "I_joint_sd": float(np.std(ijs)),
            "pct_of_H": 100*float(np.mean(ijs))/hy,
            **{f"{a}_mean": float(np.mean(acc[a])) for a in acc},
            **{f"{a}_sd": float(np.std(acc[a])) for a in acc},
            "syn_p": float(((syn_null_f >= syn0).sum()+1)/(n_eff+1)) if n_eff else float("nan"),
            "I_joint_p": float(((ij_null_f >= ij0).sum()+1)/(n_eff+1)) if n_eff else float("nan"),
            "n_perm_effective": n_eff, "n_perm_dropped": n_drop,
            "gate_pass": bool(float(((ij_null >= ij0).sum()+1)/(n_used+1)) <= 0.05),
            "converged_all_seeds": bool(all(conv))}

out = []
for db, mod, cc in DBS:
    for target in TARGETS:
        for measure in ("broja", "ccs"):
            t0 = time.time()
            r = run(db, mod, cc, target, measure)
            out.append(r)
            print(f"[{db}] {target:14s} {measure:5s} n={r['n']:7,d} "
                  f"I={r['I_joint_mean']*1e3:6.2f}±{r['I_joint_sd']*1e3:4.2f} "
                  f"({r['pct_of_H']:5.2f}%H) syn={r['syn_mean']:5.1f}±{r['syn_sd']:4.1f} "
                  f"gate={'P' if r['gate_pass'] else 'F'} ({time.time()-t0:.0f}s)", flush=True)
res = pd.DataFrame(out)

codebooks: 10 = 2 databases x 5 seeds, each fit on the full cohort


[MIMIC-IV] mortality      broja n= 64,623 I= 39.57±0.45 (11.63%H) syn= 31.5± 2.7 gate=P (5s)


[MIMIC-IV] mortality      ccs   n= 64,623 I= 39.57±0.45 (11.63%H) syn= 28.1± 2.1 gate=P (54s)


[MIMIC-IV] gender_female  broja n= 64,623 I= 30.55±3.42 ( 4.46%H) syn= 36.3± 5.1 gate=P (5s)


[MIMIC-IV] gender_female  ccs   n= 64,623 I= 30.55±3.42 ( 4.46%H) syn= 23.6± 2.5 gate=P (54s)


...

[eICU-CRD] gender_female  ccs   n=120,224 I= 25.42±3.11 ( 3.69%H) syn= 10.8± 1.0 gate=P (55s)


[eICU-CRD] age_ge65       broja n=120,266 I= 27.60±1.79 ( 3.98%H) syn=  9.3± 0.6 gate=P (5s)


[eICU-CRD] age_ge65       ccs   n=120,266 I= 27.60±1.79 ( 3.98%H) syn= 20.2± 0.8 gate=P (55s)


[eICU-CRD] race_binary    broja n=106,676 I=  8.28±0.55 ( 2.25%H) syn= 20.7± 2.3 gate=P (5s)


[eICU-CRD] race_binary    ccs   n=106,676 I=  8.28±0.55 ( 2.25%H) syn= 22.0± 1.7 gate=P (55s)


## 3. Headline table, matched panel

In [7]:
b = res[res.measure == "broja"]
pm = lambda m, s: [f"{a:.1f} ± {x:.1f}" for a, x in zip(m, s)]
show = pd.DataFrame({
    "db": b.db, "target": b.target, "n": b.n,
    "I x1e-3": [f"{m*1e3:.2f} ± {s*1e3:.2f}" for m, s in zip(b.I_joint_mean, b.I_joint_sd)],
    "% of H(Y)": [f"{v:.2f}" for v in b.pct_of_H],
    "redundant": pm(b.red_mean, b.red_sd), "unq. value": pm(b.u_val_mean, b.u_val_sd),
    "unq. structure": pm(b.u_str_mean, b.u_str_sd), "synergistic": pm(b.syn_mean, b.syn_sd),
    "syn p": [f"{v:.4f}" for v in b.syn_p], "gate": b.gate_pass})
print(show.to_string(index=False))

      db        target      n      I x1e-3 % of H(Y)  redundant unq. value unq. structure synergistic  syn p  gate
MIMIC-IV     mortality  64623 39.57 ± 0.45     11.63 24.5 ± 2.1 39.8 ± 2.6      4.2 ± 2.0  31.5 ± 2.7 0.0010  True
MIMIC-IV gender_female  64623 30.55 ± 3.42      4.46 18.1 ± 2.1 45.4 ± 6.9      0.1 ± 0.1  36.3 ± 5.1 0.0010  True
MIMIC-IV      age_ge65  64623 22.55 ± 1.37      3.26 11.9 ± 1.1 61.3 ± 2.2     -0.0 ± 0.0  26.8 ± 1.3 0.0233  True
MIMIC-IV   race_binary  48361 12.01 ± 0.84      3.22 23.1 ± 2.4 33.0 ± 3.6     12.5 ± 3.8  31.4 ± 1.8 0.0010  True
eICU-CRD     mortality 120266 21.25 ± 1.63      6.79 20.0 ± 1.4 49.1 ± 4.5      0.1 ± 0.2  30.8 ± 3.0 0.0010  True
eICU-CRD gender_female 120224 25.42 ± 3.11      3.69  1.8 ± 0.1 87.7 ± 1.3     -0.0 ± 0.0  10.5 ± 1.3 0.0010  True
eICU-CRD      age_ge65 120266 27.60 ± 1.79      3.98  9.3 ± 0.4 81.5 ± 0.8     -0.0 ± 0.0   9.3 ± 0.6 0.0010  True
eICU-CRD   race_binary 106676  8.28 ± 0.55      2.25  9.9 ± 1.1 69.4 ± 3.1     -

## 4. The comparison the whole thing was for

With the panel matched and the protocol identical, any remaining difference
between the two databases is attributable to the data source — health system,
practice, case mix — and not to how many analytes each contributes.

In [8]:
side = b.pivot(index="target", columns="db",
               values=["pct_of_H", "red_mean", "u_val_mean", "u_str_mean", "syn_mean"])
print(side.to_string(float_format=lambda x: f"{x:.2f}"))
print("\nsynergy gap (MIMIC - eICU), percentage points:")
g = b.pivot(index="target", columns="db", values="syn_mean")
print((g["MIMIC-IV"] - g["eICU-CRD"]).to_string(float_format=lambda x: f"{x:+.2f}"))

              pct_of_H          red_mean          u_val_mean          u_str_mean          syn_mean         
db            MIMIC-IV eICU-CRD MIMIC-IV eICU-CRD   MIMIC-IV eICU-CRD   MIMIC-IV eICU-CRD MIMIC-IV eICU-CRD
target                                                                                                     
age_ge65          3.26     3.98    11.88     9.25      61.35    81.46      -0.00    -0.00    26.78     9.29
gender_female     4.46     3.69    18.15     1.82      45.42    87.67       0.09    -0.00    36.34    10.52
mortality        11.63     6.79    24.54    19.97      39.78    49.10       4.16     0.14    31.51    30.79
race_binary       3.22     2.25    23.09     9.89      32.95    69.39      12.52    -0.00    31.44    20.72

synergy gap (MIMIC - eICU), percentage points:
target
age_ge65        +17.49
gender_female   +25.82
mortality        +0.72
race_binary     +10.72


## 5. Save

In [9]:
payload = {
    "design": ("Matched 12-analyte panel on both databases. Magnesium dropped from MIMIC-IV's "
               "core set rather than added to eICU's, because eICU's magnesium coverage (0.5836) "
               "would have cost 46.1% of its cohort and selected a 1.47x higher-mortality "
               "subpopulation. MIMIC's coverage is 0.9618, so dropping it GAINS 2,168 patients."),
    "panel": sorted(PANEL12_EICU),
    "supersedes": ("the per-database core-panel headline (MIMIC 13 analytes / eICU 6), "
                   "whose notebook is archived under archive/v2_k7_era: comparing "
                   "databases at different channel dimensionality confounded the "
                   "contrast this file exists to make"),
    "selection_signature": {
        "note": ("Ordering magnesium marks a sicker patient in BOTH databases; the difference in "
                 "cohort impact is coverage, not practice."),
        "eICU": {"coverage": 0.5836, "mortality_drawn": 0.1086, "mortality_not_drawn": 0.0741},
        "MIMIC": {"coverage": 0.9618, "mortality_drawn": 0.1084, "mortality_not_drawn": 0.0650}},
    "protocol": {"k": K, "grid": "fixed pre-collapse", "seeds": SEEDS,
                 "measures": ["broja", "ccs"]},
    "plateau_check": plateau.to_dict(orient="records"),
    "cells": res.to_dict(orient="records"),
}
with open("../results/headline_matched_panel12.json", "w") as f:
    json.dump(payload, f, indent=2, default=float)
print("wrote results/headline_matched_panel12.json")
print(f"gate: {int(res.gate_pass.sum())}/{len(res)} pass;  converged: {bool(res.converged_all_seeds.all())}")

wrote results/headline_matched_panel12.json
gate: 16/16 pass;  converged: True
